## Getting Started with PyTorch on Cloud TPUs

This notebook will show you how to:

* Install PyTorch/XLA on Colab, which lets you use PyTorch with TPUs.
* Run basic PyTorch functions on TPUs, like creating and adding tensors.
* Run PyTorch modules and autograd on TPUs.
* Run PyTorch networks on TPUs.

PyTorch/XLA is a package that lets PyTorch connect to Cloud TPUs and use TPU cores as devices. Colab provides a free Cloud TPU system (a remote CPU host + four TPU chips with two cores each) and installing PyTorch/XLA only takes a couple minutes.

To use PyTorch on Cloud TPUs in your own Colab notebook you can copy this one, or copy the setup cell below and configure your Colab environment to use TPUs.




<h3>  &nbsp;&nbsp;Use Colab Cloud TPU&nbsp;&nbsp; <a href="https://cloud.google.com/tpu/"><img valign="middle" src="https://raw.githubusercontent.com/GoogleCloudPlatform/tensorflow-without-a-phd/master/tensorflow-rl-pong/images/tpu-hexagon.png" width="50"></a></h3>

* On the main menu, click Runtime and select **Change runtime type**. Set "v6e-1 TPU" as the hardware accelerator.
* The cell below makes sure you have access to a TPU on Colab.


## Creating and Manipulating Tensors on TPUs

PyTorch uses Cloud TPUs just like it uses CPU or CUDA devices, as the next few cells will show. Each core of a Cloud TPU is treated as a different PyTorch  device.




In [1]:
# imports pytorch
import torch

# imports the torch_xla package
import torch_xla
import torch_xla.core.xla_model as xm

As mentioned above, the PyTorch/XLA package (torch_xla) lets PyTorch use TPU devices. The `xla_device()` function returns the TPU's "default" core as a device. This lets PyTorch creates tensors on TPUs:

In [2]:
# Creates a random tensor on xla:1 (a Cloud TPU core)
dev = xm.xla_device()
t1 = torch.ones(3, 3, device = dev)
print(t1)

tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]], device='xla:0')


/tmp/ipykernel_694/2816699506.py:2: DeprecationWarning: Use torch_xla.device instead
  dev = xm.xla_device()


See the documentation at http://pytorch.org/xla/ for a description of all public PyTorch/XLA functions. Here `xm.xla_device()` acquired the first Cloud TPU core ('xla:1'). Other cores can be directly acquired, too:

It is recommended that you use functions like `xm.xla_device()` over directly specifying TPU cores.

Tensors on TPUs can be manipulated like any other PyTorch tensor. The following cell adds, multiplies, and matrix multiplies two tensors on a TPU core:

In [4]:
a = torch.randn(2, 2, device = dev)
b = torch.randn(2, 2, device = dev)
print(a + b)
print(b * 2)
print(torch.matmul(a, b))

tensor([[ 0.3355, -1.4628],
        [-1.4656,  0.3196]], device='xla:0')
tensor([[-0.5731,  0.0157],
        [-0.5087, -0.7656]], device='xla:0')
tensor([[ 0.1946,  0.5671],
        [ 0.1691, -0.2787]], device='xla:0')


This next cell runs a 1D convolution on a TPU core:

In [5]:
# Creates random filters and inputs to a 1D convolution
filters = torch.randn(33, 16, 3, device = dev)
inputs = torch.randn(20, 16, 50, device = dev)
torch.nn.functional.conv1d(inputs, filters)

tensor([[[ -0.6830,  -0.9985,  -0.0458,  ...,   7.7178,   7.5858,  -2.8771],
         [ -7.0560,  -5.0480,  -5.7157,  ...,   3.3338, -14.5295,   6.9752],
         [  4.1409,   6.0195,  -2.5633,  ...,  17.0196,  -1.2035,  -1.4079],
         ...,
         [-11.4857, -18.0728,  -3.9573,  ...,  10.9967,  -0.9941,   9.2025],
         [ -3.4787,   0.5595,   7.0203,  ..., -10.2438,  -1.5381, -10.3243],
         [ -0.5093,   1.1812,  -1.9923,  ...,   8.4637,   4.8760,  10.0171]],

        [[-14.2448,  -2.2107,  12.6160,  ..., -16.3116,   0.0578,   7.7670],
         [ -0.0584,  -0.1845,   5.0131,  ...,   1.2221,  -4.2938,  -6.2338],
         [  2.6041,   8.5230,   9.8945,  ...,   7.3431,  -5.4920,   0.6292],
         ...,
         [ -0.7077,   1.9682, -14.5271,  ..., -11.0380, -13.6374,   4.9134],
         [  6.7691,   6.4779,   5.5757,  ...,  -0.5240,   0.6940,  -1.9311],
         [-11.4140,   0.9266,  -8.5170,  ...,   7.7713,  -1.0311,   2.7246]],

        [[-11.0244,   0.9708,   6.4415,  ...

And tensors can be transferred between CPU and TPU. In the following cell, a tensor on the CPU is copied to a TPU core, and then copied back to the CPU again. Note that PyTorch makes copies of tensors when transferring them across devices, so `t_cpu` and `t_cpu_again` are different tensors.



In [6]:
# Creates a tensor on the CPU (device='cpu' is unnecessary and only added for clarity)
t_cpu = torch.randn(2, 2, device='cpu')
print(t_cpu)

t_tpu = t_cpu.to(dev)
print(t_tpu)

t_cpu_again = t_tpu.to('cpu')
print(t_cpu_again)

tensor([[-1.8446,  0.6499],
        [-0.9607, -0.0697]])
tensor([[-1.8446,  0.6499],
        [-0.9607, -0.0697]], device='xla:0')
tensor([[-1.8446,  0.6499],
        [-0.9607, -0.0697]])


## Running PyTorch modules and autograd on TPUs

Modules and autograd are fundamental PyTorch components.

In PyTorch, every stateful function is a module. Modules are Python classes augmented with metadata that lets PyTorch understand how to use them in a neural network. For example, linear layers are modules, as are entire networks. Since modules are stateful, they can be placed on devices, too. PyTorch/XLA lets us place them on TPU cores:


In [7]:
# Creates a linear module
fc = torch.nn.Linear(5, 2, bias=True)

# Copies the module to the XLA device (the first Cloud TPU core)
fc = fc.to(dev)

# Creates a random feature tensor
features = torch.randn(3, 5, device=dev, requires_grad=True)

# Runs and prints the module
output = fc(features)
print(output)

tensor([[-0.8349,  0.5356],
        [ 0.4170, -0.2348],
        [-0.2441, -0.2130]], device='xla:0', grad_fn=<AddmmBackward0>)


Autograd is the system PyTorch uses to populate the gradients of weights in a neural network. See [here](https://pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html#sphx-glr-beginner-blitz-autograd-tutorial-py) for details about PyTorch's autograd. When a module is run on a TPU core, its gradients are also populated on the same TPU core by autograd. The following cell demonstrates this:

In [8]:
output.backward(torch.ones_like(output))
print(fc.weight.grad)

tensor([[ 0.1177, -2.0664,  0.4219,  1.4258, -0.1000],
        [ 0.1177, -2.0664,  0.4219,  1.4258, -0.1000]], device='xla:0')


## Running PyTorch networks on TPUs

As mentioned above, PyTorch networks are also modules, and so they're run in the same way. The following cell runs a relatively simple PyTorch network from the [PyTorch tutorial docs](https://pytorch.org/tutorials/beginner/blitz/neural_networks_tutorial.html#sphx-glr-beginner-blitz-neural-networks-tutorial-py) on a TPU core:

In [9]:
import torch.nn as nn
import torch.nn.functional as F

# Simple example network from
# https://pytorch.org/tutorials/beginner/blitz/neural_networks_tutorial.html#sphx-glr-beginner-blitz-neural-networks-tutorial-py
class Net(nn.Module):

    def __init__(self):
        super(Net, self).__init__()
        # 1 input image channel, 6 output channels, 3x3 square convolution
        # kernel
        self.conv1 = nn.Conv2d(1, 6, 3)
        self.conv2 = nn.Conv2d(6, 16, 3)
        # an affine operation: y = Wx + b
        self.fc1 = nn.Linear(16 * 6 * 6, 120)  # 6*6 from image dimension
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        # Max pooling over a (2, 2) window
        x = F.max_pool2d(F.relu(self.conv1(x)), (2, 2))
        # If the size is a square you can only specify a single number
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = x.view(-1, self.num_flat_features(x))
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

    def num_flat_features(self, x):
        size = x.size()[1:]  # all dimensions except the batch dimension
        num_features = 1
        for s in size:
            num_features *= s
        return num_features


# Places network on the default TPU core
net = Net().to(dev)

# Creates random input on the default TPU core
input = torch.randn(1, 1, 32, 32, device=dev)

# Runs network
out = net(input)
print(out)

tensor([[-0.0212,  0.0208,  0.1196, -0.0550, -0.0634, -0.0444, -0.0226, -0.0005,
          0.0665, -0.0362]], device='xla:0', grad_fn=<AddmmBackward0>)


As in the previous snippets, running PyTorch on a TPU just requires specifying a TPU core as a device.